# 11.09 · Code execution in sandboxes

**Module 11 · Deep Agents** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 1.5–2 h |
| 📋 Prerequisites | Lessons 11.03–11.04, 11.08; optional: Docker Desktop or Docker Engine. |
| 🔑 Keys | none for 🧪/🐳 cells · optional 💸 cell ≈ 8–12K tokens (off by default) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** what a sandbox backend adds (`execute`) and why it must be isolated.
2. **Tell apart** `LocalShellBackend` (no isolation) and a real sandbox.
3. **Describe** how `BaseSandbox` builds every file tool on top of `execute()`.
4. **Run** an agent's code in a locked-down Docker container, and keep secrets out of it.

## 📚 Official reading (verified 2026-10-16, `deepagents` 0.7.26)

| Page | Read it for |
|---|---|
| [Sandboxes](https://docs.langchain.com/oss/python/deepagents/sandboxes) | Providers, lifecycle (thread vs assistant scope), the two planes of file access, security |
| [Backends](https://docs.langchain.com/oss/python/deepagents/backends) | Where `LocalShellBackend` fits, and its warning |
| [Sandbox integrations](https://docs.langchain.com/oss/python/integrations/sandboxes) | Managed providers (LangSmith, Daytona, E2B, Modal, Runloop, …) |

> 🐳 **Docker is optional.** 🐳 cells use your local Docker to run code in a container; they need no API key and spend no tokens. Without Docker they print ⏭️ Skipped.

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "deepagents>=0.7.23,<0.8" "langchain-google-genai>=4.4.1,<5" "langchain-groq>=1.1.3,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("deepagents", "langchain",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/11-deep-agents/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GOOGLE_API_KEY', 'GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm11-sandboxes'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.deep import deep_key_ready, deep_model_spec, fallback_specs

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")

DEEP_MODEL = deep_model_spec("main")              # MASTERY_DEEP_MODEL (course default: Gemini free tier)
SUBAGENT_MODEL = deep_model_spec("subagent")      # MASTERY_DEEP_SUBAGENT_MODEL
HAVE_DEEP_KEY = deep_key_ready(DEEP_MODEL)        # the key for THAT provider (GOOGLE_API_KEY for google_genai:)
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m11-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("deep model:", DEEP_MODEL, "| subagents:", SUBAGENT_MODEL)
print("fallbacks :", fallback_specs(DEEP_MODEL) or "none (optional: MASTERY_FALLBACK_MODELS in .env)")
if not HAVE_DEEP_KEY:
    print(f"\n⚠️ No key for {DEEP_MODEL.split(':')[0]}: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🧪 Helpers.
import sys
import tempfile

from langchain_core.messages import AIMessage

from mastery.deep import DockerSandbox, build_deep_agent, docker_available, run_stats
from mastery.testing import ScriptedChatModel


def calls(*items) -> AIMessage:
    return AIMessage("", tool_calls=[{"name": n, "args": a, "id": i} for n, a, i in items])


def user(text: str) -> dict:
    return {"messages": [{"role": "user", "content": text}]}


PY = f'"{sys.executable}"'          # absolute interpreter path: works in sh and in Windows cmd
HAVE_DOCKER = docker_available()
print("Docker available:", HAVE_DOCKER, "(optional: the 🐳 cells skip without it)")

---
## Part 1 · What a sandbox adds

In [ ]:
# 🧪 A sandbox backend adds ONE tool: execute. Same agent, two backends.
from deepagents.backends import LocalShellBackend

script = ScriptedChatModel(responses=[AIMessage("ok")])
build_deep_agent(script, fallbacks=False).invoke(user("hi"))
print("StateBackend      :", script.bound_tools)

script = ScriptedChatModel(responses=[AIMessage("ok")])
build_deep_agent(script, fallbacks=False, backend=LocalShellBackend(root_dir=tempfile.mkdtemp(), inherit_env=False)).invoke(user("hi"))
print("LocalShellBackend :", script.bound_tools)

**You should now see** the same tool list, plus `execute` with the shell backend. The harness checks whether the backend implements `SandboxBackendProtocol` and only then offers `execute`.

## Part 2 · A shell is not a sandbox

In [ ]:
# 🧪 LocalShellBackend is NOT a sandbox: virtual_mode guards the FILE tools, not the shell.
local = LocalShellBackend(root_dir=tempfile.mkdtemp(prefix="m11-shell-"), inherit_env=False)
try:
    local.read("/../../outside.txt")
except ValueError as err:
    print("read_file outside root :", err)
probe = local.execute(PY + ' -c "import os; print(sorted(os.listdir(os.sep))[:3])"')
print("execute lists host root:", probe.output.strip(), "| exit", probe.exit_code)
keys = local.execute(PY + ' -c "import os; print(sorted(k for k in os.environ if k.endswith(chr(95) + chr(75) + chr(69) + chr(89))))"')
print("API-key env vars seen  :", keys.output.strip(), "(inherit_env=False keeps them out)")

**You should now see** the file tool refuse to leave its root, while `execute` happily lists your machine's root directory: `virtual_mode` only protects the file tools. `inherit_env=False` at least keeps your API keys out of the shell.

> ⚠️ **Production note: never give an agent `LocalShellBackend` on a machine that matters.** The Backends page says it plainly: "No isolation—use only in controlled development environments." Prompt injection turns "summarise this README" into "run this command". In production: a container or managed sandbox per thread, no host mounts, no secrets inside, network off unless needed, and human approval (`interrupt_on={"execute": True}`) for anything irreversible.

## Part 3 · How a sandbox provider works

In [ ]:
# 🧪 How BaseSandbox works: every file tool becomes a command sent to execute().
from deepagents.backends.protocol import ExecuteResponse
from deepagents.backends.sandbox import BaseSandbox


class RecordingSandbox(BaseSandbox):
    """Records commands instead of running them (portable: nothing executes)."""
    def __init__(self):
        self.commands = []

    @property
    def id(self):
        return "recording"

    def execute(self, command, *, timeout=None):
        self.commands.append(command)
        return ExecuteResponse(output="", exit_code=0, truncated=False)

    def upload_files(self, files):
        raise NotImplementedError

    def download_files(self, paths):
        raise NotImplementedError


box = RecordingSandbox()
box.ls("/workspace")
box.read("/workspace/report.md")
for command in box.commands:
    print(f"{len(command):>5} chars | {command[:110]!r}")

**What just happened?** `ls` and `read` became shell/Python commands for `execute()`. That's why a provider only implements four things (`execute`, `upload_files`, `download_files`, `id`) and inherits `read_file`, `edit_file`, `grep`, … for free. It also means the image needs `sh` and `python3`.

## Part 4 · 🐳 A Docker sandbox

In [ ]:
# 🐳 The course DockerSandbox: one locked-down container per run. Read the isolation flags.
print(" ".join(DockerSandbox().run_args()))

**What just happened?** `mastery.deep.DockerSandbox` (about 100 lines, read it in `src/mastery/deep/sandbox.py`) runs `--network none`, drops all Linux capabilities, caps memory/CPU/processes, mounts nothing from your disk and passes no environment variables. The container is removed when the `with` block ends.

In [ ]:
# 🐳 (needs Docker, no tokens) A scripted agent writes a script and runs it INSIDE the container.
if HAVE_DOCKER:
    code_text = "import statistics\nprint(statistics.mean([3, 5, 10]))\n"
    script = ScriptedChatModel(responses=[
        calls(("write_file", {"file_path": "/workspace/mean.py", "content": code_text}, "w")),
        calls(("execute", {"command": "python /workspace/mean.py && whoami && (python -c 'import socket; socket.create_connection((\"1.1.1.1\", 53), 2)' 2>/dev/null && echo network-on || echo no-network)"}, "x")),
        AIMessage("The mean is 6.")])
    with DockerSandbox() as sandbox:                     # first run pulls python:3.12-slim (~50 MB)
        out = build_deep_agent(script, backend=sandbox, fallbacks=False).invoke(user("Mean of 3, 5, 10?"))
    print([m.content for m in out["messages"] if m.type == "tool"][-1])
else:
    print("⏭️ Skipped: Docker is not available (install Docker Desktop / Engine to run the 🐳 cells).")

**You should now see** (with Docker) `6.0`, a user name from inside the container, and `no-network`. Without Docker, ⏭️ Skipped.

> ⚠️ **Production note: secrets stay outside.** "Never put secrets inside a sandbox": a context-injected agent can read and exfiltrate them. Keep authenticated actions in **tools that run on your side** (the agent calls them by name and never sees the key), and use a managed provider for TTLs, snapshots and audit logs.

## Part 5 · 💸🐳 Real analysis in a container

> 💸 Calls your deep model (`MASTERY_DEEP_MODEL`, default Gemini free tier). Rate-limited to one call every 4 s and capped per run. Without the key it prints `⏭️ Skipped`.

In [ ]:
# 💸🐳 LIVE (opt-in): a real model analyses a CSV with code it writes and runs in Docker. ≈ 8–12K tokens.
RUN_LIVE = False
CSV = "month,signups\nJan,120\nFeb,135\nMar,90\nApr,160\nMay,210\n"
if RUN_LIVE and HAVE_DEEP_KEY and HAVE_DOCKER:
    with DockerSandbox() as sandbox:
        sandbox.upload_files([("/workspace/signups.csv", CSV.encode())])
        live = build_deep_agent(backend=sandbox, model_calls=10, tool_calls=12,
                                system_prompt="You are a data analyst. Use execute to run Python in /workspace.")
        result = live.invoke(user("Which month had the biggest growth in signups vs the previous month? Compute it."))
    print("tools :", [m.name for m in result["messages"] if m.type == "tool"])
    print("answer:", result["messages"][-1].text[:300])
    print("stats :", run_stats(result))
elif not RUN_LIVE:
    print("⏭️ Skipped: set RUN_LIVE = True to try it (needs Docker; ≈ 8–12K tokens).")
elif not HAVE_DOCKER:
    print("⏭️ Skipped: Docker is not available.")
else:
    print(f"⏭️ Skipped: no key for {DEEP_MODEL} (see the setup cell).")

**You should now see** (when enabled) `write_file`/`execute` calls and an answer naming **April** (+70 signups vs March; May is only +50); check it by hand: the agent's arithmetic is now code you can read.

## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| No `execute` tool | Backend isn't a sandbox (State/Store/Filesystem) | Use a sandbox backend |
| `docker: command not found` / daemon error | Docker not installed or not running | Start Docker Desktop; `docker info` must work |
| `pip install` fails inside | `--network none` | Bake dependencies into the image, or enable network deliberately |
| Files written by `execute` invisible to `read_file` | Different working directory | Use absolute paths (`/workspace/...`) |
| Containers left running | No `with` / crash before cleanup | Use the context manager; `docker ps --filter name=m11-sandbox` |

## 🏋️ Exercises

1. **(Easy)** Add `interrupt_on={"execute": True}` and a checkpointer to the Docker cell. Approve the command by hand.
2. **(Medium)** Run `box.edit(...)` and `box.grep(...)` on `RecordingSandbox` and look at the commands they generate.
3. **(Medium)** Create `DockerSandbox(network=True)` and compare `run_args()`. When would you allow network, and what would you add?
4. **(Stretch)** Upload a CSV with `upload_files`, let the scripted agent run a script on it, and fetch a generated file back with `download_files`.

<details>
<summary><b>Show solutions</b></summary>

**1.** Pass `checkpointer=InMemorySaver()` and `interrupt_on={"execute": True}`, then resume with `Command(resume={"decisions": [{"type": "approve"}]})` as in Lesson 11.08.

**2.** `box.edit("/workspace/a.py", "x", "y")` and `box.grep("TODO", path="/workspace")` record long commands: Python replacement scripts and a `grep` invocation.

**3.** `--network none` disappears. Allow it only for a task that needs it (e.g. `pip install` from a mirror), and add an egress allowlist or proxy; never with secrets inside.

**4.**
```python
with DockerSandbox() as sb:
    sb.upload_files([("/workspace/in.csv", b"a\n1\n2\n")])
    sb.execute("python -c \"print(sum(int(x) for x in open('/workspace/in.csv').read().split()[1:]))\" > /workspace/out.txt")
    print(sb.download_files(["/workspace/out.txt"])[0].content)   # b'3\n'
```
</details>

## 🔎 Docs drill (5 min)

Sandbox lifecycle: what is the difference between a thread-scoped and an assistant-scoped sandbox, and what do you have to add for the second one?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Sandboxes · Lifecycle and scoping](https://docs.langchain.com/oss/python/deepagents/sandboxes).
2. Thread-scoped (default): each conversation gets its own sandbox, reused on follow-up turns and gone when the thread ends or its TTL expires. Assistant-scoped: every thread of the same assistant reuses one sandbox, so files and packages persist across conversations.
3. For assistant scope, configure a TTL, snapshots or cleanup so disk and memory don't grow without bound.

</details>

## ✅ Checkpoint

- [ ] I can explain what `execute` adds and why it needs isolation.
- [ ] I know `LocalShellBackend` is not a sandbox.
- [ ] I can explain how `BaseSandbox` builds the file tools.
- [ ] I know where secrets go (tools outside the sandbox) and why.
- [ ] Restart & Run All passes without keys (and without Docker); outputs cleared before saving.

## 🔎 Research checkpoint

On [Sandbox integrations](https://docs.langchain.com/oss/python/integrations/sandboxes), pick one managed provider and note its free tier, network controls and TTL options today. Log it with the date in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- A sandbox backend = file tools + `execute`; only `execute()` is provider-specific.
- `LocalShellBackend` is a supervised-dev convenience, not isolation.
- Isolate per thread, no network by default, no secrets inside, approve risky commands.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 11-10-streaming-to-a-ui.ipynb** → [`11-10-streaming-to-a-ui.ipynb`](11-10-streaming-to-a-ui.ipynb).

[Module 11 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)